[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Carlo-Broschi/statistics-python-for-students/blob/main/01_Python%E5%9F%BA%E7%A4%8E/06_pandas%E5%85%A5%E9%96%80.ipynb)

> Colabで実行可。最初に「① セットアップ」セルを実行（Colabはデータを自動生成、ローカルは何もしない）。

In [ ]:
# === ① セットアップ（最初に実行してください）===============================
# Colabなどデータが無い環境では、教材と同一の合成データを自動生成します。
# ローカル/Codespacesで data/ がある場合は何もしません（再現性のため乱数seedは固定）。
import os
if not os.path.exists('../data/students_scores.csv'):   # データが無い環境（Colab等）か判定
    os.makedirs('/content/nb', exist_ok=True); os.chdir('/content/nb')  # ../data を使えるよう作業場所を作る
    os.makedirs('../data', exist_ok=True)               # データ保存先フォルダを作る
    import numpy as np, pandas as pd                     # 数値計算と表データのライブラリ
    D = '../data'; rng = np.random.default_rng(42)       # 保存先と、結果を固定する乱数生成器
    # --- 生徒120人の成績データを作る ---
    n = 120; cls = rng.choice(['A','B','C'], n)          # 人数とクラス
    math = np.clip(rng.normal(65,15,n),0,100).round().astype(int)        # 数学（0〜100点）
    eng = np.clip(0.6*math+rng.normal(28,10,n),0,100).round().astype(int) # 英語（数学と相関）
    jp = np.clip(rng.normal(70,12,n),0,100).round().astype(int)          # 国語
    hrs = np.clip(rng.normal(1.5,0.8,n)+math/100,0,None).round(1)        # 勉強時間
    pd.DataFrame({'生徒ID':[f'S{i:03d}' for i in range(1,n+1)],'クラス':cls,
        '数学':math,'英語':eng,'国語':jp,'勉強時間':hrs}).to_csv(f'{D}/students_scores.csv',index=False,encoding='utf-8-sig')
    # --- 30日間の天気データ ---
    days = pd.date_range('2026-06-01', periods=30)       # 6月の30日分の日付
    temp = (24+4*np.sin(np.arange(30)/5)+rng.normal(0,2,30)).round(1)    # 気温
    rain = np.clip(rng.gamma(1.2,4,30)*(rng.random(30)<0.4),0,None).round(1)  # 降水量
    pd.DataFrame({'日付':days.strftime('%Y-%m-%d'),'気温':temp,'降水量':rain}).to_csv(f'{D}/weather.csv',index=False,encoding='utf-8-sig')
    # --- BtoB商談400件 ---
    m = 400; inds = rng.choice(['製造','IT','小売','医療','金融','建設'],m,p=[.25,.2,.18,.12,.15,.1])  # 業界
    chs = ['展示会','Web広告','メルマガ','紹介','テレアポ']; ch = rng.choice(chs,m,p=[.2,.3,.2,.15,.15])  # 獲得チャネル
    deal = (rng.lognormal(13,0.7,m)).round(-3).astype(int)   # 商談金額
    wr = {'展示会':.35,'Web広告':.18,'メルマガ':.12,'紹介':.45,'テレアポ':.15}  # チャネル別の受注率
    won = np.array([rng.random()<wr[c] for c in ch])         # 受注したか（チャネルで確率を変える）
    dt = pd.to_datetime('2026-01-01')+pd.to_timedelta(rng.integers(0,180,m),unit='D')  # 受注日
    pd.DataFrame({'商談ID':[f'D{i:04d}' for i in range(1,m+1)],'受注日':dt.strftime('%Y-%m-%d'),
        '業界':inds,'獲得チャネル':ch,'商談金額':deal,'担当者':rng.choice(['田中','鈴木','佐藤','高橋'],m),
        '受注':won.astype(int)}).to_csv(f'{D}/sales_btob.csv',index=False,encoding='utf-8-sig')
    # --- 月×チャネルのマーケ実績 ---
    months = pd.date_range('2026-01-01',periods=6,freq='MS').strftime('%Y-%m')  # 6か月分
    base = {'展示会':2000,'Web広告':12000,'メルマガ':6000,'紹介':800,'テレアポ':1500}  # 表示回数の目安
    ctr = {'展示会':.08,'Web広告':.03,'メルマガ':.05,'紹介':.2,'テレアポ':.12}   # クリック率
    cvr = {'展示会':.12,'Web広告':.04,'メルマガ':.06,'紹介':.25,'テレアポ':.09}   # 獲得率
    cpc = {'展示会':30,'Web広告':80,'メルマガ':5,'紹介':0,'テレアポ':200}        # クリック単価
    rows = []
    for mo in months:                                    # 月ごと
        for c in chs:                                    # チャネルごとに実績を作る
            imp = int(base[c]*rng.uniform(0.8,1.3)); clk = int(imp*ctr[c]*rng.uniform(0.8,1.2))
            cv = int(clk*cvr[c]*rng.uniform(0.7,1.3)); cost = int(clk*cpc[c]*rng.uniform(0.9,1.1))
            rows.append([mo,c,imp,clk,cv,cost])
    pd.DataFrame(rows,columns=['月','チャネル','表示回数','クリック数','獲得数','費用']).to_csv(f'{D}/web_marketing.csv',index=False,encoding='utf-8-sig')
    # --- A/Bテストの申込ログ ---
    ab = []
    for grp,p,size in [('A_青ボタン',0.082,2400),('B_緑ボタン',0.104,2380)]:  # 2群の申込率と人数
        for c in (rng.random(size)<p): ab.append([grp,int(c)])   # 各訪問が申込んだか
    pd.DataFrame(ab,columns=['グループ','申込']).sample(frac=1,random_state=1).reset_index(drop=True).to_csv(f'{D}/ab_test.csv',index=False,encoding='utf-8-sig')
    print('✅ 合成データを生成しました（Colab環境）')
else:
    print('✅ data/ を検出しました（ローカル/Codespaces）')

# 06. pandas入門 〜表データを自在に扱う〜

**pandas** は、Excelのような「表（テーブル）」をPythonで扱う道具。
CSVを読み込み、選び・絞り・集計する——統計・マーケのノートの中心スキルです。

**ゴール**：CSVを読み込んで、列の取り出し・行の絞り込み・グループ集計ができる。

In [ ]:
import pandas as pd   # 表データ用ライブラリ pandas を pd として読み込む
import numpy as np    # 数値計算用 NumPy を np として読み込む

## 1. Series と DataFrame

- **Series**：1列のデータ（ラベル付きの配列）
- **DataFrame**：複数列の表

In [ ]:
# ラベル(index)付きの1列データ Series を作る
s = pd.Series([60, 75, 90], index=['数学', '英語', '国語'])
print(s)                  # Series全体を表示
print('英語の点:', s['英語'])  # ラベル'英語'の値を取り出す

In [ ]:
# 辞書から複数列の表 DataFrame を作る
df = pd.DataFrame({
    '名前': ['あおい', 'はると', 'ゆい'],
    '数学': [90, 60, 75],
    '英語': [70, 85, 80],
})
df   # 表を表示

## 2. CSVを読み込む

実際のデータは `read_csv` で読みます。教材の生徒データを使います。

In [ ]:
df = pd.read_csv('../data/students_scores.csv')  # CSVファイルを読み込む
df.head()       # 先頭5行を確認

### 使うデータ：`students_scores.csv`（架空の生徒120人の成績）
1行＝生徒1人。数学・英語・国語は0〜100点、勉強時間は1日あたりの時間です。

| 生徒ID | クラス | 数学 | 英語 | 国語 | 勉強時間 |
|---|---|---|---|---|---|
| S001 | A | 52 | 58 | 49 | 2.0 |
| S002 | C | 80 | 79 | 74 | 3.4 |
| S003 | B | 40 | 65 | 91 | 2.0 |

（下のセルの `df.head()` で先頭5行を実際に確認できます）

In [ ]:
print('行数・列数:', df.shape)        # (行数, 列数)
print('列名:', list(df.columns))      # 列名の一覧
df.describe()    # 数値列の要約（平均・標準偏差・四分位など）

## 3. 列を取り出す

`df['列名']` で1列、`df[['列1','列2']]` で複数列。

In [ ]:
print(df['数学'].head())          # '数学'列の先頭5行
print('数学の平均:', df['数学'].mean().round(1))  # '数学'列の平均
df[['数学', '英語']].head()        # 2列を選んで先頭5行

## 4. 行を絞り込む（条件フィルタ）

`df[条件]` で、条件がTrueの行だけ残します。NumPyの真偽値インデックスと同じ発想。

In [ ]:
# 数学が80点以上の行だけを残す（条件フィルタ）
good = df[df['数学'] >= 80]
print('80点以上:', len(good), '人')   # 残った行数 ＝ 人数
good.head()

In [ ]:
# 複数条件は & (かつ) / | (または)。各条件は () で囲む
df[(df['数学'] >= 80) & (df['英語'] >= 80)].head()  # 数学80以上 かつ 英語80以上

## 5. 新しい列をつくる

In [ ]:
df['合計'] = df['数学'] + df['英語'] + df['国語']  # 3教科の合計を新しい列に
df['合否'] = np.where(df['数学'] >= 60, '合格', '不合格')  # 数学60以上で合否を判定
df[['生徒ID', '数学', '合計', '合否']].head()   # 必要な列だけ先頭5行

## 6. グループごとに集計（groupby）

**統計・マーケで最頻出**。「クラスごとの平均点」のような集計が1行で書けます。

In [ ]:
# クラスごとに'数学'の平均を計算
print(df.groupby('クラス')['数学'].mean().round(1))

# クラスごとに複数の集計をまとめて行う
df.groupby('クラス').agg(
    人数=('生徒ID', 'count'),    # 人数（件数）
    数学平均=('数学', 'mean'),    # 数学の平均
    英語平均=('英語', 'mean'),    # 英語の平均
).round(1)

## 7. 数える・並べ替え・クロス集計

In [ ]:
print(df['合否'].value_counts())          # カテゴリごとの個数を数える
print()
print(df.sort_values('合計', ascending=False)[['生徒ID','合計']].head(3))  # 合計の高い順 上位3人
print()
print(pd.crosstab(df['クラス'], df['合否']))  # クラス×合否のクロス集計表

> `pd.cut`（階級分け）も統計でよく使います → `02_統計_4級/01` で登場します。

---
## 練習問題

**問1.** `students_scores.csv` を読み込み、`英語` の平均と最高点を求めよう。

**問2.** `国語` が80点以上の生徒だけを絞り込み、何人いるか数えよう。

**問3.** クラスごとの `勉強時間` の平均を `groupby` で求めよう。

**問4.** `数学` の高い順に並べ替えて、上位5人の `生徒ID` と `数学` を表示しよう。

In [ ]:
# 準備
import pandas as pd                               # pandasを読み込む
df = pd.read_csv('../data/students_scores.csv')   # データを読み込む

In [ ]:
# 問1〜4


> **解答例は別ページにまとめています**（ネタバレ防止）。
> 自分で解いてから **[06_pandas入門 の解答例を開く](https://github.com/Carlo-Broschi/statistics-python-for-students/blob/main/%E8%A7%A3%E7%AD%94%E9%9B%86/01_Python%E5%9F%BA%E7%A4%8E/06_pandas%E5%85%A5%E9%96%80.md)**